# rpm sweep: onset of Taylor vortices

Post-processing for `studies/rpm_sweep.sh`. Each case in `cases/rpm_sweep/rpm_<speed>` is an
axisymmetric r-z wedge of the bob and cup. The notebook reads the latest time of every case
and looks at the axial velocity in the annular gap, because Taylor vortices show up as
alternating axial flow along the gap.

The bob bottom drives circulation near the bottom end, so cells are present there at every
speed. The question is how far up the gap the cells reach as the speed increases.

Set `SWEEP` below to the sweep folder. The path is relative to this notebook (`post/`).

In [ ]:
%matplotlib inline
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from cfdtools import post

SWEEP = Path("../cases/rpm_sweep")
SPEEDS_FILE = Path("../studies/rpm_sweep.speeds")
SHOW = [600, 700, 750, 800]         # speeds drawn in section 3

post.use_style()

## 1. Read the cases

The speeds come from the speeds file, in the order they were run. A case counts as converged
if `log.simpleFoam` contains `SIMPLE solution converged`. The iteration count is the number
of `Time =` lines in the same log.

In [ ]:
def read_speeds(path):
    text = [line.split("//")[0].strip() for line in Path(path).read_text().splitlines()]
    return [int(t) for t in text if t]


def log_summary(case):
    log = (case / "log.simpleFoam").read_text()
    iterations = len(re.findall(r"^Time = ", log, flags=re.M))
    return iterations, "SIMPLE solution converged" in log


speeds = [s for s in read_speeds(SPEEDS_FILE) if (SWEEP / f"rpm_{s}").is_dir()]
cases = {}
for s in speeds:
    path = SWEEP / f"rpm_{s}"
    mesh, t = post.read_case(path)
    iterations, converged = log_summary(path)
    cases[s] = dict(path=path, mesh=mesh, iterations=iterations, converged=converged)
    print(f"rpm_{s}: {iterations} iterations, {'converged' if converged else 'NOT converged'}")

## 2. Geometry and expected onset

`case_parameters` gives the SI values the case was built with. The annulus is taken to span
the top `H` of the mesh, because the top of the mesh is the fluid surface.

The critical speed is estimated with the narrow-gap Taylor number
`Ta = omega^2 Ri d^3 / nu^2`, which becomes unstable at about 1708 for an infinitely long
gap. The gap here is not narrow (`Ri/Ro = 0.87`) and has end walls, so this is a rough guide.

In [ ]:
p = post.case_parameters(cases[speeds[0]]["path"])
Ri, Ro, H, nu = (float(p[k]) for k in ("Ri", "Ro", "H", "nu"))
d = Ro - Ri
z_top = max(c["mesh"].bounds[5] for c in cases.values())
z_bottom = z_top - H

TA_CRIT = 1708.0
omega_c = np.sqrt(TA_CRIT * nu**2 / (Ri * d**3))
rpm_c = omega_c * 60 / (2 * np.pi)

print(f"Ri = {Ri*1e3} mm, Ro = {Ro*1e3} mm, gap = {d*1e3:.3f} mm, H = {H*1e3} mm")
print(f"annulus spans z = {z_bottom*1e3:.2f} to {z_top*1e3:.2f} mm")
print(f"estimated critical speed: {rpm_c:.0f} rpm")

## 3. Axial velocity in the gap

One panel per speed, showing the annulus only. All panels share one symmetric colour range, so
equal colours mean equal velocities. Blue and red are axial flow in opposite directions, so
alternating bands along the gap are vortex cells.

In [ ]:
shown = [s for s in SHOW if s in cases]
vmax = max(np.abs(cases[s]["mesh"].cell_data["U"][:, 2]).max() for s in shown)

fig, axes = plt.subplots(1, len(shown), figsize=(1.3 * len(shown) + 1, 5), sharey=True)
for ax, s in zip(np.atleast_1d(axes), shown):
    pc = post.plot_field(ax, cases[s]["mesh"], "U", which="z", normal="y",
                         cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_aspect("auto")
    ax.set_xlim(Ri, Ro)
    ax.set_ylim(z_bottom, z_top)
    ax.set_xticks([])
    ax.set_title(f"{s} rpm")
np.atleast_1d(axes)[0].set_ylabel("z [m]")
fig.colorbar(pc, ax=axes, label=r"$u_z$ [m/s]", shrink=0.6)
plt.show()

## 4. Axial velocity along the middle of the gap

`u_z` sampled along a line at the gap centre, from the bottom of the annulus to the surface.
The sign changes along the line mark the cell boundaries. Only the part below the surface
slip wall is shown, and the colours follow the speed.

In [ ]:
r_mid = 0.5 * (Ri + Ro)
colours = post.sequential_cmap()(np.linspace(0.25, 1.0, len(speeds)))

fig, ax = plt.subplots(figsize=(4, 5))
for s, colour in zip(speeds, colours):
    mesh = cases[s]["mesh"]
    y_mid = 0.5 * (mesh.bounds[2] + mesh.bounds[3])
    dist, uz = post.sample_line(mesh, (r_mid, y_mid, z_bottom), (r_mid, y_mid, z_top), "U", which="z")
    ax.plot(uz * 1e3, (z_bottom + dist) * 1e3, color=colour, label=f"{s} rpm")
ax.axvline(0, color=post.INK, linewidth=0.5)
ax.set_xlabel(r"$u_z$ at mid-gap [mm/s]")
ax.set_ylabel("z [mm]")
ax.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
plt.show()

## 5. Cell strength against speed

For each speed, the largest `|u_z|` over the annulus cells in the upper half of the gap
(`z` above the middle of the annulus), where the bottom-driven circulation has decayed. Above
onset, the amplitude grows as the square root of the distance from the critical speed, so the
square of the amplitude should rise linearly with speed and cross zero at the critical speed.

In [ ]:
def upper_amplitude(mesh):
    c = post.cell_centres(mesh)
    r = np.hypot(c[:, 0], c[:, 1])
    inside = (r > Ri) & (r < Ro) & (c[:, 2] > z_top - 0.5 * H)
    return np.abs(mesh.cell_data["U"][inside, 2]).max()


amp = np.array([upper_amplitude(cases[s]["mesh"]) for s in speeds])
rpm = np.array(speeds, dtype=float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))
ax1.plot(rpm, amp * 1e3, marker="o", color=post.SERIES[0])
ax1.set_xlabel("speed [rpm]")
ax1.set_ylabel(r"max $|u_z|$, upper half [mm/s]")

ax2.plot(rpm, (amp * 1e3) ** 2, marker="o", color=post.SERIES[0])
ax2.set_xlabel("speed [rpm]")
ax2.set_ylabel(r"(max $|u_z|$)$^2$ [(mm/s)$^2$]")
for ax in (ax1, ax2):
    ax.axvline(rpm_c, color=post.INK, linestyle="--", linewidth=1)
ax1.text(rpm_c, ax1.get_ylim()[1], " estimate", va="top", ha="left", fontsize="small")
fig.tight_layout()
plt.show()

## 6. Straight-line fit to the amplitude squared

A straight line through the highest speeds, extended to zero, gives an estimate of the
critical speed from the simulation. Choose how many of the highest speeds to use with
`N_FIT`. The fit is only meaningful if those speeds are clearly above onset, so check against
the plot above before trusting it.

In [ ]:
N_FIT = 3

x, y = rpm[-N_FIT:], (amp[-N_FIT:] * 1e3) ** 2
slope, intercept = np.polyfit(x, y, 1)
rpm_fit = -intercept / slope
print(f"fit through {x.astype(int).tolist()} rpm: critical speed about {rpm_fit:.0f} rpm "
      f"(narrow-gap estimate {rpm_c:.0f} rpm)")

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(rpm, (amp * 1e3) ** 2, marker="o", linestyle="none", color=post.SERIES[0])
xx = np.linspace(rpm_fit, rpm.max(), 50)
ax.plot(xx, slope * xx + intercept, color=post.INK, linestyle="--", linewidth=1)
ax.axvline(rpm_c, color=post.INK, linewidth=0.5)
ax.set_xlabel("speed [rpm]")
ax.set_ylabel(r"(max $|u_z|$)$^2$ [(mm/s)$^2$]")
plt.show()

## 7. Iterations to converge

Convergence slows down near onset, so the iteration count peaks close to the critical speed.
A case that did not converge is drawn with an open marker.

In [ ]:
iters = [cases[s]["iterations"] for s in speeds]
done = [cases[s]["converged"] for s in speeds]

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(speeds, iters, color=post.SERIES[0], linewidth=1)
for s, n, ok in zip(speeds, iters, done):
    ax.plot(s, n, marker="o", color=post.SERIES[0], markerfacecolor=post.SERIES[0] if ok else "white")
ax.set_xlabel("speed [rpm]")
ax.set_ylabel("iterations")
ax.set_yscale("log")
plt.show()